# Mô hình 3: LSTM Deep Learning

LSTM được chọn vì PM2.5 là chuỗi thời gian có phụ thuộc ngắn hạn và dài hạn.
Mô hình nhận 48 giờ quá khứ và dự báo vector PM2.5 cho 24 giờ tiếp theo.

## Bài toán học máy

Mục tiêu là dự báo nồng độ bụi mịn `PM2.5` sau 24 giờ cho Hà Nội, TP.HCM và Đà Nẵng
dựa trên dữ liệu chất lượng không khí, khí tượng, thời gian, đặc trưng trễ và trung bình trượt.

Vì đây là chuỗi thời gian, dữ liệu được chia tuần tự theo `target_time`:
Train 70%, Validation 15%, Test 15%. Validation dùng để theo dõi/tinh chỉnh mô hình;
Test chỉ dùng để đánh giá cuối cùng.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, Input, LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")
tf.keras.utils.set_random_seed(42)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "models":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "pm25_training_data.csv"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

SEQ_LEN = 48
PRED_LEN = 24
STRIDE = 6
BATCH_SIZE = 256
EPOCHS = 12
FEATURES = ["pm25", "temp", "humidity", "wind_speed", "pressure", "pm10"]
TARGET_IDX = FEATURES.index("pm25")
PM25_BINS = [-np.inf, 12.0, 35.4, 55.4, 150.4, 250.4, np.inf]
PM25_LABELS = ["Good", "Moderate", "USG", "Unhealthy", "Very Unhealthy", "Hazardous"]

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def pm25_category(values):
    return pd.cut(values, bins=PM25_BINS, labels=PM25_LABELS, include_lowest=True)

## Chuẩn bị dữ liệu sequence và chuẩn hóa không rò rỉ

In [ ]:
df = pd.read_csv(DATA_PATH)
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values(["city", "datetime"]).reset_index(drop=True)
df = df.dropna(subset=FEATURES).copy()

unique_times = pd.Series(df["datetime"].sort_values().unique())
train_cut = unique_times.iloc[int(len(unique_times) * 0.70)]
val_cut = unique_times.iloc[int(len(unique_times) * 0.85)]

scaler = MinMaxScaler()
scaler.fit(df.loc[df["datetime"] < train_cut, FEATURES])

X_train, y_train, meta_train = [], [], []
X_val, y_val, meta_val = [], [], []
X_test, y_test, meta_test = [], [], []

for city, city_df in df.groupby("city", sort=False):
    city_df = city_df.sort_values("datetime").reset_index(drop=True)
    scaled = scaler.transform(city_df[FEATURES])
    raw_values = city_df[FEATURES].to_numpy()
    times = pd.to_datetime(city_df["datetime"])

    last_start = len(city_df) - SEQ_LEN - PRED_LEN + 1
    for start in range(0, max(0, last_start), STRIDE):
        input_end = start + SEQ_LEN - 1
        target_start = start + SEQ_LEN
        target_end = target_start + PRED_LEN - 1
        target_time = times.iloc[target_end]

        X_seq = scaled[start : start + SEQ_LEN]
        y_seq = scaled[target_start : target_start + PRED_LEN, TARGET_IDX]
        meta = {
            "city": city,
            "datetime": times.iloc[input_end],
            "target_time": target_time,
            "actual_pm25": float(raw_values[target_end, TARGET_IDX]),
        }

        if target_time < train_cut:
            X_train.append(X_seq); y_train.append(y_seq); meta_train.append(meta)
        elif target_time < val_cut:
            X_val.append(X_seq); y_val.append(y_seq); meta_val.append(meta)
        else:
            X_test.append(X_seq); y_test.append(y_seq); meta_test.append(meta)

X_train = np.asarray(X_train, dtype=np.float32)
y_train = np.asarray(y_train, dtype=np.float32)
X_val = np.asarray(X_val, dtype=np.float32)
y_val = np.asarray(y_val, dtype=np.float32)
X_test = np.asarray(X_test, dtype=np.float32)
y_test = np.asarray(y_test, dtype=np.float32)
meta_test = pd.DataFrame(meta_test)

split_table = pd.DataFrame(
    {
        "Tập dữ liệu": ["Train", "Validation", "Test"],
        "Số sequence": [len(X_train), len(X_val), len(X_test)],
        "Tỷ lệ (%)": [
            len(X_train) / (len(X_train) + len(X_val) + len(X_test)) * 100,
            len(X_val) / (len(X_train) + len(X_val) + len(X_test)) * 100,
            len(X_test) / (len(X_train) + len(X_val) + len(X_test)) * 100,
        ],
        "Mô tả": [
            f"target_time < {train_cut}",
            f"{train_cut} <= target_time < {val_cut}",
            f"target_time >= {val_cut}",
        ],
    }
)
print("Bảng 1. Tỷ lệ chia sequence cho LSTM")
display(split_table)
print("Input shape:", X_train.shape[1:], "| Output horizon:", PRED_LEN)

## Kiến trúc và cấu hình huấn luyện

- Kiến trúc: Input(48, 6) -> LSTM(64) -> Dropout(0.2) -> LSTM(32) -> Dropout(0.2) -> Dense(16, ReLU) -> Dense(24, Linear).
- Hàm kích hoạt: LSTM dùng `tanh` và cổng `sigmoid`; Dense ẩn dùng `ReLU`; output dùng tuyến tính.
- Loss: MSE, phù hợp cho hồi quy PM2.5.
- Optimizer: Adam, learning rate 0.001.
- Scheduler: ReduceLROnPlateau, giảm learning rate khi Validation Loss không cải thiện.
- Early stopping: dừng sớm và khôi phục trọng số tốt nhất để hạn chế overfitting.

In [ ]:
model_lstm = Sequential(
    [
        Input(shape=(SEQ_LEN, len(FEATURES))),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(PRED_LEN),
    ]
)

model_lstm.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"],
)
model_lstm.summary()

arch_rows = []
for layer in model_lstm.layers:
    activation = getattr(layer, "activation", None)
    activation_name = activation.__name__ if activation is not None else "-"
    if layer.__class__.__name__ == "LSTM":
        activation_name = "tanh / sigmoid gates"
    arch_rows.append(
        {
            "layer": layer.name,
            "type": layer.__class__.__name__,
            "output_shape": str(layer.output.shape),
            "activation": activation_name,
            "params": layer.count_params(),
        }
    )
arch_table = pd.DataFrame(arch_rows)
arch_table.loc[len(arch_table)] = {
    "layer": "TOTAL",
    "type": "-",
    "output_shape": "-",
    "activation": "-",
    "params": model_lstm.count_params(),
}
arch_table.to_csv(RESULTS_DIR / "lstm_architecture.csv", index=False, encoding="utf-8-sig")

print("Bảng 2. Kiến trúc LSTM và số lượng tham số")
display(arch_table)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.axis("off")
layers = [
    ("Input", f"{SEQ_LEN}x{len(FEATURES)}", "-"),
    ("LSTM", "64 units", "tanh/sigmoid"),
    ("Dropout", "rate=0.2", "-"),
    ("LSTM", "32 units", "tanh/sigmoid"),
    ("Dropout", "rate=0.2", "-"),
    ("Dense", "16 units", "ReLU"),
    ("Output", f"{PRED_LEN} units", "Linear"),
]
xs = np.linspace(0.08, 0.92, len(layers))
for idx, (name, shape, act) in enumerate(layers):
    ax.text(
        xs[idx],
        0.56,
        f"{name}\n{shape}\n{act}",
        ha="center",
        va="center",
        fontsize=9,
        bbox={"boxstyle": "round,pad=0.35", "facecolor": "#f5f7fb", "edgecolor": "#4c566a"},
    )
    if idx < len(layers) - 1:
        ax.annotate(
            "",
            xy=(xs[idx + 1] - 0.055, 0.56),
            xytext=(xs[idx] + 0.055, 0.56),
            arrowprops={"arrowstyle": "->", "lw": 1.2, "color": "#4c566a"},
        )
ax.text(0.5, 0.18, f"Total trainable parameters: {model_lstm.count_params():,}", ha="center", fontsize=10)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "lstm_architecture.png", dpi=180)
plt.show()

Hình 1. Sơ đồ kiến trúc LSTM dự báo PM2.5 đa bước 24 giờ.

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
]

history = model_lstm.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,
    callbacks=callbacks,
    verbose=1,
)

history_df = pd.DataFrame(history.history)
history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))
history_df.to_csv(RESULTS_DIR / "lstm_training_history.csv", index=False)

## Learning curves theo epoch

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["loss"], label="Train Loss")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="Validation Loss")
axes[0].set_title("LSTM Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_df["epoch"], history_df["mae"], label="Train MAE")
axes[1].plot(history_df["epoch"], history_df["val_mae"], label="Validation MAE")
axes[1].set_title("LSTM MAE")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Scaled MAE")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "lstm_learning_curves.png", dpi=180)
plt.show()

Hình 2. Learning curves của LSTM trên Train và Validation. Nếu Validation Loss giảm rồi đi ngang
trong khi Train Loss tiếp tục giảm, mô hình có dấu hiệu bắt đầu overfitting; Dropout và Early Stopping
được dùng để giảm hiện tượng này.

## Đánh giá Test và phân tích lỗi

In [ ]:
def inverse_pm25_matrix(values_scaled):
    n_samples, pred_len = values_scaled.shape
    dummy = np.zeros((n_samples * pred_len, len(FEATURES)))
    dummy[:, TARGET_IDX] = values_scaled.reshape(-1)
    inv = scaler.inverse_transform(dummy)[:, TARGET_IDX]
    return inv.reshape(n_samples, pred_len)

train_pred_scaled = model_lstm.predict(X_train, batch_size=BATCH_SIZE, verbose=0)
val_pred_scaled = model_lstm.predict(X_val, batch_size=BATCH_SIZE, verbose=0)
test_pred_scaled = model_lstm.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

train_actual = inverse_pm25_matrix(y_train)[:, -1]
val_actual = inverse_pm25_matrix(y_val)[:, -1]
test_actual = inverse_pm25_matrix(y_test)[:, -1]
train_pred = inverse_pm25_matrix(train_pred_scaled)[:, -1]
val_pred = inverse_pm25_matrix(val_pred_scaled)[:, -1]
test_pred = inverse_pm25_matrix(test_pred_scaled)[:, -1]

metrics = pd.DataFrame(
    [
        {
            "model": "LSTM",
            "scope": "All cities",
            "train_rmse_ug_m3": rmse(train_actual, train_pred),
            "val_rmse_ug_m3": rmse(val_actual, val_pred),
            "test_rmse_ug_m3": rmse(test_actual, test_pred),
            "test_mae_ug_m3": float(mean_absolute_error(test_actual, test_pred)),
            "test_r2": float(r2_score(test_actual, test_pred)),
        }
    ]
)
metrics.to_csv(RESULTS_DIR / "lstm_metrics.csv", index=False, encoding="utf-8-sig")
print("Bảng 3. Chỉ số đánh giá LSTM, đơn vị lỗi: µg/m³")
display(metrics)

actual_cat = pm25_category(test_actual)
pred_cat = pm25_category(np.clip(test_pred, 0, None))
cm = confusion_matrix(actual_cat, pred_cat, labels=PM25_LABELS)
cm_df = pd.DataFrame(cm, index=PM25_LABELS, columns=PM25_LABELS)
cm_df.to_csv(RESULTS_DIR / "lstm_confusion_matrix.csv", encoding="utf-8-sig")

class_summary = pd.DataFrame(
    [
        {
            "model": "LSTM",
            "accuracy": float(accuracy_score(actual_cat, pred_cat)),
            "precision_macro": float(precision_score(actual_cat, pred_cat, labels=PM25_LABELS, average="macro", zero_division=0)),
            "recall_macro": float(recall_score(actual_cat, pred_cat, labels=PM25_LABELS, average="macro", zero_division=0)),
            "f1_macro": float(f1_score(actual_cat, pred_cat, labels=PM25_LABELS, average="macro", zero_division=0)),
            "precision_weighted": float(precision_score(actual_cat, pred_cat, labels=PM25_LABELS, average="weighted", zero_division=0)),
            "recall_weighted": float(recall_score(actual_cat, pred_cat, labels=PM25_LABELS, average="weighted", zero_division=0)),
            "f1_weighted": float(f1_score(actual_cat, pred_cat, labels=PM25_LABELS, average="weighted", zero_division=0)),
        }
    ]
)
class_report = pd.DataFrame(
    classification_report(actual_cat, pred_cat, labels=PM25_LABELS, output_dict=True, zero_division=0)
).T
class_summary.to_csv(RESULTS_DIR / "lstm_classification_summary.csv", index=False, encoding="utf-8-sig")
class_report.to_csv(RESULTS_DIR / "lstm_classification_report.csv", encoding="utf-8-sig")

print("Bảng 4. Chỉ số cảnh báo LSTM sau khi phân nhóm PM2.5")
display(class_summary)
print("Bảng 5. Ma trận nhầm lẫn LSTM sau khi ánh xạ PM2.5")
display(cm_df)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_title("LSTM confusion matrix by PM2.5 category")
ax.set_xlabel("Predicted category")
ax.set_ylabel("Actual category")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "lstm_confusion_matrix.png", dpi=180)
plt.show()

errors = meta_test.copy()
errors["actual_pm25"] = test_actual
errors["predicted_pm25"] = test_pred
errors["abs_error_ug_m3"] = np.abs(test_actual - test_pred)
errors["actual_category"] = actual_cat.astype(str)
errors["predicted_category"] = pred_cat.astype(str)
errors["error_hypothesis"] = np.where(
    errors["predicted_pm25"] < errors["actual_pm25"],
    "Có xu hướng đánh giá thấp đỉnh ô nhiễm hoặc biến động bất thường.",
    "Có xu hướng đánh giá cao do chuỗi quá khứ chưa phản ánh trạng thái khuếch tán hiện tại.",
)
top_errors = errors.sort_values("abs_error_ug_m3", ascending=False).head(20)
top_errors.to_csv(RESULTS_DIR / "lstm_top_error_cases.csv", index=False, encoding="utf-8-sig")
print("Bảng 6. Các trường hợp sai lớn nhất của LSTM, đơn vị lỗi: µg/m³")
display(top_errors.head(10))

Hình 3. Ma trận nhầm lẫn của LSTM sau khi ánh xạ PM2.5 thực tế/dự báo sang nhóm rủi ro.